# 👸 SITA — Getting Started

This notebook walks through how to use **SITA** (Standardized Infrastructure for the Training of Adapters) both from the **CLI** and **programmatically** in Python.

We'll cover:
1. **Installation & Setup**
2. **Running an experiment from the CLI**
3. **Running an experiment programmatically**
4. **Inspecting the registries**
5. **Swapping adapters / trainers via code**
6. **Writing a custom adapter**
7. **Comparing multiple PEFT methods in one go**

---
## 1. Installation

In [ ]:
# Install SITA from the repo
!pip install "sita[all] @ git+https://github.com/<your-username>/SITA.git" -q

---
## 2. CLI Usage (Quick Reference)

The simplest way to run an experiment is via the command line:

```bash
# Run a LoRA experiment
sita configs/lora_causal_lm.yaml

# Run a QLoRA experiment
sita configs/qlora_causal_lm.yaml

# List all registered components
sita --list-registry

# Verbose logging
sita configs/lora_causal_lm.yaml -v
```

Below, we'll do everything **from Python** instead.

---
## 3. Running an Experiment Programmatically

In [ ]:
import logging

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s: %(message)s",
    datefmt="%H:%M:%S",
)

In [ ]:
from sita.core.config import load_config
from sita.runner import _import_builtins, run_experiment

# Step 1: Import built-in components so they register themselves
_import_builtins()

# Step 2: Load a YAML config
config = load_config("../configs/lora_causal_lm.yaml")
print(config)

In [ ]:
# Step 3: Run the full pipeline (load → adapt → train → evaluate)
metrics = run_experiment(config)
print("\n✅ Done! Metrics:", metrics)

---
## 4. Inspecting the Registries

SITA uses a registry pattern — every model, adapter, dataset, trainer, and evaluator registers itself with a string key. You can inspect what's available at any time.

In [ ]:
from sita.core.registry import (
    MODEL_REGISTRY,
    ADAPTER_REGISTRY,
    DATASET_REGISTRY,
    TRAINER_REGISTRY,
    EVALUATOR_REGISTRY,
)

# Make sure builtins are imported first (already done above)
print("📦 Models:    ", MODEL_REGISTRY.list())
print("🔧 Adapters:  ", ADAPTER_REGISTRY.list())
print("📂 Datasets:  ", DATASET_REGISTRY.list())
print("🏋️ Trainers:  ", TRAINER_REGISTRY.list())
print("📊 Evaluators:", EVALUATOR_REGISTRY.list())

In [ ]:
# You can also retrieve a specific component class:
LoRAClass = ADAPTER_REGISTRY.get("lora")
print(f"LoRA adapter class: {LoRAClass}")
print(f"Is 'qlora' registered? {'qlora' in ADAPTER_REGISTRY}")

---
## 5. Swapping Adapters & Trainers via Code

Instead of editing YAML files, you can build the config entirely in Python.

In [ ]:
from sita.core.config import (
    ExperimentConfig,
    ModelConfig,
    AdapterConfig,
    DatasetConfig,
    TrainingConfig,
    TrainerConfig,
    EvalConfig,
)

# Building a config purely in Python is possible, without having to define the YAML
my_config = ExperimentConfig(
    experiment_name="lora-from-notebook",
    seed=123,
    model=ModelConfig(
        name="hf_causal_lm",
        pretrained="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
        kwargs={"torch_dtype": "float16", "device_map": "auto"},
    ),
    adapter=AdapterConfig(
        name="lora",  # ← swap to "qlora" or "prefix_tuning" here!
        kwargs={
            "r": 8,
            "lora_alpha": 16,
            "lora_dropout": 0.05,
            "target_modules": ["q_proj", "v_proj"],
            "task_type": "CAUSAL_LM",
        },
    ),
    dataset=DatasetConfig(
        name="hf_dataset",
        kwargs={"path": "tatsu-lab/alpaca", "text_field": "text", "max_length": 512},
    ),
    training=TrainingConfig(
        output_dir="./output/lora-notebook",
        num_epochs=1,
        batch_size=4,
        learning_rate=2e-4,
        gradient_accumulation_steps=4,
        fp16=True,
        logging_steps=10,
    ),
    trainer=TrainerConfig(name="hf_trainer"),
    evaluation=EvalConfig(name="loss", kwargs={"batch_size": 8}),
)

print(my_config)

In [ ]:
# Run it!
metrics = run_experiment(my_config)
print("\n✅ Results:", metrics)

### 5a. Using the Custom Training Loop

Just swap the trainer name to `"custom_loop"` and pass extra kwargs:

In [ ]:
from dataclasses import replace

# Take the same config, but swap the trainer
custom_loop_config = replace(
    my_config,
    experiment_name="lora-custom-loop-notebook",
    trainer=TrainerConfig(
        name="custom_loop",
        kwargs={"optimizer": "adamw", "scheduler": "cosine", "log_grad_norm": True},
    ),
    training=replace(
        my_config.training,
        output_dir="./output/lora-custom-loop-notebook",
        bf16=True,
        fp16=False,
    ),
)

print(f"Trainer: {custom_loop_config.trainer.name}")
# metrics = run_experiment(custom_loop_config)  # uncomment to run

### 5b. Using Prefix Tuning Instead of LoRA

In [ ]:
prefix_config = replace(
    my_config,
    experiment_name="prefix-tuning-notebook",
    adapter=AdapterConfig(
        name="prefix_tuning",
        kwargs={"num_virtual_tokens": 20, "task_type": "CAUSAL_LM"},
    ),
    training=replace(my_config.training, output_dir="./output/prefix-notebook"),
)

print(f"Adapter: {prefix_config.adapter.name}")
print(f"  kwargs: {prefix_config.adapter.kwargs}")
# metrics = run_experiment(prefix_config)  # uncomment to run

---
## 6. Writing a Custom Adapter

To add your own PEFT method, you only need one file. Here's an example that wraps IA³ from PEFT:

In [ ]:
# You can define and register a custom adapter right here in the notebook

from torch import nn
from peft import IA3Config, PeftModel, get_peft_model

from sita.core.base_adapter import BaseAdapter
from sita.core.config import AdapterConfig
from sita.core.registry import ADAPTER_REGISTRY


@ADAPTER_REGISTRY.register("ia3")
class IA3Adapter(BaseAdapter):
    """IA³ (Infused Adapter by Inhibiting and Amplifying Inner Activations)."""

    def apply(self, model: nn.Module, config: AdapterConfig) -> nn.Module:
        ia3_config = IA3Config(**config.kwargs)
        model = get_peft_model(model, ia3_config)
        model.print_trainable_parameters()
        return model

    def save(self, model: nn.Module, path: str) -> None:
        model.save_pretrained(path)

    def load(self, model: nn.Module, path: str) -> nn.Module:
        return PeftModel.from_pretrained(model, path)


# Verify it's now in the registry
print("Adapters:", ADAPTER_REGISTRY.list())
assert "ia3" in ADAPTER_REGISTRY

In [ ]:
# Now use it just like any built-in adapter
ia3_config = replace(
    my_config,
    experiment_name="ia3-notebook",
    adapter=AdapterConfig(
        name="ia3",
        kwargs={
            "target_modules": ["k_proj", "v_proj", "down_proj"],
            "feedforward_modules": ["down_proj"],
            "task_type": "CAUSAL_LM",
        },
    ),
    training=replace(my_config.training, output_dir="./output/ia3-notebook"),
)

print(f"Adapter: {ia3_config.adapter.name}")
# metrics = run_experiment(ia3_config)  # uncomment to run

---
## 7. Comparing Multiple PEFT Methods

One of SITA's main goals is to make A/B comparisons easy. Here's a pattern for running multiple experiments and collecting results:

In [ ]:
from dataclasses import replace

# Define the adapter configs to compare
adapter_sweep = {
    "lora-r8": AdapterConfig(
        name="lora",
        kwargs={"r": 8, "lora_alpha": 16, "target_modules": ["q_proj", "v_proj"], "task_type": "CAUSAL_LM"},
    ),
    "lora-r16": AdapterConfig(
        name="lora",
        kwargs={"r": 16, "lora_alpha": 32, "target_modules": ["q_proj", "v_proj"], "task_type": "CAUSAL_LM"},
    ),
    "prefix-20": AdapterConfig(
        name="prefix_tuning",
        kwargs={"num_virtual_tokens": 20, "task_type": "CAUSAL_LM"},
    ),
}

# Base config (shared across all experiments)
base = ExperimentConfig(
    seed=42,
    model=ModelConfig(
        name="hf_causal_lm",
        pretrained="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
        kwargs={"torch_dtype": "float16", "device_map": "auto"},
    ),
    dataset=DatasetConfig(
        name="hf_dataset",
        kwargs={"path": "tatsu-lab/alpaca", "text_field": "text", "max_length": 512},
    ),
    training=TrainingConfig(
        num_epochs=1, batch_size=4, learning_rate=2e-4,
        gradient_accumulation_steps=4, fp16=True, logging_steps=50,
    ),
    trainer=TrainerConfig(name="hf_trainer"),
    evaluation=EvalConfig(name="loss", kwargs={"batch_size": 8}),
)

print(f"Will compare {len(adapter_sweep)} adapter configurations:")
for name in adapter_sweep:
    print(f"  • {name}")

In [ ]:
# Run all experiments and collect results
# ⚠️ This will take a while! Comment out the loop below for a dry run.

results = {}

for exp_name, adapter_cfg in adapter_sweep.items():
    print(f"\n{'='*60}")
    print(f"Running: {exp_name}")
    print(f"{'='*60}")

    exp_config = replace(
        base,
        experiment_name=exp_name,
        adapter=adapter_cfg,
        training=replace(base.training, output_dir=f"./output/{exp_name}"),
    )

    # metrics = run_experiment(exp_config)   # ← uncomment to actually train
    # results[exp_name] = metrics

    # Placeholder for dry run
    results[exp_name] = {"eval_loss": 0.0, "perplexity": 0.0}

print("\n✅ All experiments done!")

In [ ]:
# Display results as a table
import pandas as pd

df = pd.DataFrame(results).T
df.index.name = "experiment"
df = df.sort_values("eval_loss")
print(df.to_markdown())
df

---
## 8. Step-by-Step Pipeline (Advanced)

If you need more control, you can call each pipeline stage individually instead of using `run_experiment()`.

In [ ]:
from sita.core.registry import MODEL_REGISTRY, ADAPTER_REGISTRY, DATASET_REGISTRY

# Load the model
model_loader = MODEL_REGISTRY.get("hf_causal_lm")()
model, tokenizer = model_loader.load(
    ModelConfig(
        name="hf_causal_lm",
        pretrained="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
        kwargs={"torch_dtype": "float16", "device_map": "auto"},
    )
)
print(f"Model loaded: {type(model).__name__}")
print(f"Tokenizer vocab: {tokenizer.vocab_size}")

In [ ]:
# 2. Apply the adapter
adapter = ADAPTER_REGISTRY.get("lora")()
model = adapter.apply(
    model,
    AdapterConfig(
        name="lora",
        kwargs={"r": 16, "lora_alpha": 32, "target_modules": ["q_proj", "v_proj"], "task_type": "CAUSAL_LM"},
    ),
)

# Check trainable parameters
params = adapter.get_trainable_params(model)
print(f"Trainable: {params['trainable_params']:,} / {params['total_params']:,} ({params['trainable_pct']:.2f}%)")

In [ ]:
# 3. Load Dataset
dataset_loader = DATASET_REGISTRY.get("hf_dataset")()
train_ds, eval_ds = dataset_loader.load(
    DatasetConfig(
        name="hf_dataset",
        kwargs={"path": "tatsu-lab/alpaca", "text_field": "text", "max_length": 512},
    ),
    tokenizer,
)
print(f"Train samples: {len(train_ds)}")
print(f"Eval samples:  {len(eval_ds) if eval_ds else 'N/A'}")

In [ ]:
# 4. Train (using HF Trainer) 
from sita.core.registry import TRAINER_REGISTRY

trainer = TRAINER_REGISTRY.get("hf_trainer")()

training_cfg = TrainingConfig(
    output_dir="./output/step-by-step",
    num_epochs=1,
    batch_size=4,
    learning_rate=2e-4,
    gradient_accumulation_steps=4,
    fp16=True,
    logging_steps=10,
)

# model = trainer.train(model, tokenizer, train_ds, eval_ds, training_cfg)  # uncomment to run

In [ ]:
# 5. Evaluate 
from sita.core.registry import EVALUATOR_REGISTRY

evaluator = EVALUATOR_REGISTRY.get("loss")()
# metrics = evaluator.evaluate(model, tokenizer, eval_ds or train_ds, batch_size=8)
# print(metrics)

In [ ]:
# 6. Save & Reload Adapter 
# adapter.save(model, "./output/step-by-step/adapter")
# reloaded_model = adapter.load(base_model, "./output/step-by-step/adapter")

---

## Summary

| What | How |
|---|---|
| Run from CLI | `sita configs/lora_causal_lm.yaml` |
| Run from Python | `run_experiment(config)` |
| Build config in code | Use `ExperimentConfig(...)` dataclasses |
| Swap adapter | Change `adapter.name` to `"lora"`, `"qlora"`, `"prefix_tuning"`, ... |
| Swap trainer | Change `trainer.name` to `"hf_trainer"` or `"custom_loop"` |
| Add new adapter | Subclass `BaseAdapter`, decorate with `@ADAPTER_REGISTRY.register(...)` |
| Compare methods | Loop over adapter configs, call `run_experiment()` |
| Step-by-step control | Call registry `.get()` for each component manually |

For more details, see the **[README](../README.md)** and the example configs in `configs/`.